# DyFHL-PA on Colab

Cross-domain access-control policy anomaly analysis with federated hypergraph sketching.

1. Set `REPO` below to your GitHub repository.
2. Run all cells. A CPU runtime is enough; the full suite takes roughly 1–2 hours on Colab CPU, `--quick` about 15 minutes.

In [ ]:
REPO = 'https://github.com/OWNER/dyfhl-pa.git'  # <- change me
!git clone -q $REPO dyfhl-pa
%cd dyfhl-pa
!pip -q install -r requirements.txt
!bash scripts/fetch_data.sh

## Pick a dataset and run detection

In [ ]:
import torch, pandas as pd
from dyfhl.policy import World, get_source
from dyfhl.model import Config, TASKS
from dyfhl.fed import run

DATASET = 'agentic'  #@param ['agentic', 'xu_stoller', 'amazon']
SEED = 0  #@param {type:'integer'}
B = 512  #@param [64, 128, 256, 512, 1024, 2048, 4096] {type:'raw'}
SIGMA = 0.0  #@param {type:'number'}

w = World(get_source(DATASET), seed=SEED)
lab = w.labels()   # exact oracle, pools all domains: evaluation only
rule = run(w, Config(seed=SEED, B=B, sigma=SIGMA, h=0), labels=lab, rule=True)
learned, clients, model, _, (P, Pq) = run(w, Config(seed=SEED, B=B, sigma=SIGMA), labels=lab, return_all=True)
pd.DataFrame({'DyFHL-PA-R': {**{t: rule[t]['f1'] for t in TASKS}, 'macro-F1': rule['macro_f1']},
              'DyFHL-PA-L': {**{t: learned[t]['f1'] for t in TASKS}, 'macro-F1': learned['macro_f1']}}).round(3)

## Hypergraph projection of the anomalies

In [ ]:
from dyfhl import viz
R, Y, Q, gap = lab
viz.embedding_projection(R, Y, P >= 0.5, method='tsne', seed=SEED).show()
viz.bipartite_view(w, R, Y, scope=0).show()

## Baselines

In [ ]:
from dyfhl.baselines import VARIANTS, rf_baseline
rows = {'RF (20% audited labels)': rf_baseline(w, lab, SEED, False), 'RF + count context': rf_baseline(w, lab, SEED, True)}
for name, ov in VARIANTS.items():
    rows[name] = run(w, Config(seed=SEED, **ov), labels=lab)
rows['DyFHL-PA-R'] = rule
pd.DataFrame({k: {**{t: v[t]['f1'] for t in TASKS}, 'macro-F1': v['macro_f1'], 'macro-AUPRC': v['macro_ap']} for k, v in rows.items()}).T.round(3)

## Policy stream (churn) with delta-maintained context

In [ ]:
from dyfhl.dynamic import run_stream
w2 = World(get_source(DATASET), seed=SEED)
s = pd.DataFrame(run_stream(w2, Config(seed=SEED, h=0, B=B), T=10, churn=0.1))
s[['t', 'f1_delta', 'f1_stale', 'bytes_delta', 'bytes_full', 'max_abs_err']].round(3)

## Reproduce every table and figure of the paper

In [ ]:
!python scripts/run_experiments.py --quick   # drop --quick for the full 5-seed suite
!python scripts/make_figures.py

## Web app
Starts the Streamlit explorer and prints a public URL.

In [ ]:
!npm -q install -g localtunnel >/dev/null 2>&1
!streamlit run app/streamlit_app.py --server.port 8501 --server.headless true &>/content/streamlit.log &
!sleep 8 && echo 'Tunnel password (your Colab IP):' && curl -s ipv4.icanhazip.com && npx localtunnel --port 8501